# Project 4: Generative AI — N-gram Language Model + Markov Chain Text Generator

From-scratch: N-gram LM with Laplace smoothing, perplexity evaluation, Markov chain generator.

In [1]:
import numpy as np
import re, random, math
from collections import defaultdict, Counter
random.seed(42)
np.random.seed(42)

# Local corpus (no network needed)
raw_texts = [
    "The quick brown fox jumps over the lazy dog near the river bank.",
    "Machine learning algorithms can identify patterns in large datasets automatically.",
    "Natural language processing enables computers to understand human speech and text.",
    "Deep learning models require significant computational resources for training purposes.",
    "Data science combines statistics mathematics and computer science for insights.",
    "Artificial intelligence is transforming industries including healthcare finance and education.",
    "Neural networks are inspired by the biological structure of the human brain.",
    "Supervised learning uses labeled training data to build predictive classification models.",
    "Unsupervised learning discovers hidden structure in unlabeled data distributions.",
    "Reinforcement learning trains agents to make decisions through reward and penalty signals.",
    "The stock market fluctuates based on investor sentiment and economic indicators.",
    "Scientists discovered a new species of deep-sea fish in the Pacific Ocean basin.",
    "Climate change poses significant risks to ecosystems and human populations worldwide.",
    "Governments around the world are investing in renewable energy infrastructure projects.",
    "The global economy faces challenges from inflation supply chain disruptions and conflict.",
    "Space exploration has expanded human knowledge about the solar system and beyond.",
    "Quantum computing promises to solve problems currently intractable for classical machines.",
    "Cybersecurity threats continue to evolve requiring constant vigilance and adaptation.",
    "The internet has fundamentally changed how people communicate and access information.",
    "Open source software enables collaborative development and accelerates technological innovation.",
] * 80  # repeat to get decent corpus

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\.\!\?\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.split()

tokens = []
for t in raw_texts:
    tokens.extend(tokenize(t))

vocab = Counter(tokens)
print("Corpus tokens  :", len(tokens))
print("Unique vocab   :", len(vocab))
print("Top 20 tokens  :", [w for w,c in vocab.most_common(20)])


Corpus tokens  : 17600
Unique vocab   : 173
Top 20 tokens  : ['the', 'and', 'to', 'learning', 'in', 'human', 'for', 'data', 'enables', 'significant', 'training', 'science', 'are', 'structure', 'of', 'has', 'quick', 'brown', 'fox', 'jumps']


In [2]:
class NGramLM:
    def __init__(self, n=3, k=0.1):
        self.n = n
        self.k = k
        self.ngrams = defaultdict(Counter)

    def train(self, tokens):
        self.vocab = set(tokens)
        self.V = len(self.vocab)
        for i in range(len(tokens) - self.n):
            ctx  = tuple(tokens[i : i + self.n - 1])
            word = tokens[i + self.n - 1]
            self.ngrams[ctx][word] += 1

    def prob(self, ctx, word):
        ctx    = tuple(ctx[-(self.n-1):])
        counts = self.ngrams[ctx]
        total  = sum(counts.values())
        return (counts[word] + self.k) / (total + self.k * self.V + 1e-9)

    def generate(self, seed, max_tokens=60, temperature=0.9):
        result = list(seed)
        for _ in range(max_tokens):
            ctx    = tuple(result[-(self.n-1):])
            counts = self.ngrams.get(ctx, {})
            if not counts:
                ctx    = tuple(result[-1:])
                counts = self.ngrams.get(ctx, {})
            if not counts:
                break
            words  = list(counts.keys())
            freqs  = np.array([counts[w] for w in words], dtype=float)
            freqs  = freqs ** (1.0 / temperature)
            freqs /= freqs.sum()
            chosen = np.random.choice(words, p=freqs)
            result.append(chosen)
            if chosen in (".", "!", "?") and len(result) > 12:
                break
        return " ".join(result)

    def perplexity(self, tokens):
        log_p, count = 0.0, 0
        for i in range(self.n - 1, len(tokens)):
            ctx  = tuple(tokens[i-(self.n-1):i])
            word = tokens[i]
            p    = self.prob(ctx, word)
            log_p += math.log(max(p, 1e-10))
            count += 1
        return math.exp(-log_p / count)

split = int(len(tokens) * 0.9)
train_tok = tokens[:split]
test_tok  = tokens[split:]
print("Train:", len(train_tok), "  Test:", len(test_tok))


Train: 15840   Test: 1760


In [3]:
print("N-gram LM Perplexity Comparison:")
lm_models = {}
for n in [2, 3, 4, 5]:
    lm = NGramLM(n=n, k=0.1)
    lm.train(train_tok)
    ppl = lm.perplexity(test_tok)
    lm_models[n] = (lm, ppl)
    print("  {}-gram   PPL={:.4f}   States={}".format(n, ppl, len(lm.ngrams)))

best_n, (best_lm, best_ppl) = min(lm_models.items(), key=lambda x: x[1][1])
print("\nBest: {}-gram  PPL={:.4f}".format(best_n, best_ppl))


N-gram LM Perplexity Comparison:
  2-gram   PPL=1.8689   States=173
  3-gram   PPL=1.2386   States=220
  4-gram   PPL=1.2386   States=220
  5-gram   PPL=1.2386   States=220

Best: 3-gram  PPL=1.2386


In [4]:
print("=== Generated Texts ===\n")
seeds_temps = [
    (["machine", "learning"], [0.5, 0.9, 1.3]),
    (["the", "global"],       [0.5, 0.9, 1.3]),
    (["natural", "language"], [0.7, 1.0]),
    (["deep", "learning"],    [0.6, 1.1]),
]
for seed, temps in seeds_temps:
    for temp in temps:
        text = best_lm.generate(seed, max_tokens=50, temperature=temp)
        print("[seed='{}' temp={}]".format(" ".join(seed), temp))
        print(text)
        print()


=== Generated Texts ===

[seed='machine learning' temp=0.5]
machine learning algorithms can identify patterns in large datasets automatically. natural language processing enables computers to understand human speech and text. deep learning models require significant computational resources for training purposes. data science combines statistics mathematics and computer science for insights. artificial intelligence is transforming industries including healthcare finance and education. neural

[seed='machine learning' temp=0.9]
machine learning algorithms can identify patterns in large datasets automatically. natural language processing enables computers to understand human speech and text. deep learning models require significant computational resources for training purposes. data science combines statistics mathematics and computer science for insights. artificial intelligence is transforming industries including healthcare finance and education. neural

[seed='machine learning' temp=1

In [5]:
class MarkovChain:
    def __init__(self, order=2):
        self.order = order
        self.trans = defaultdict(list)

    def train(self, tokens):
        for i in range(len(tokens) - self.order):
            state = tuple(tokens[i:i+self.order])
            self.trans[state].append(tokens[i+self.order])

    def generate(self, seed, length=50):
        state  = tuple(seed)
        result = list(seed)
        for _ in range(length):
            nexts = self.trans.get(state)
            if not nexts:
                break
            chosen = random.choice(nexts)
            result.append(chosen)
            state  = tuple(result[-self.order:])
        return " ".join(result)

mc = MarkovChain(order=3)
mc.train(train_tok)

print("=== Markov Chain Order-3 Samples ===\n")
for seed in [["machine","learning","algorithms"],
             ["artificial","intelligence","is"],
             ["the","global","economy"]]:
    out = mc.generate(seed, length=40)
    print("[seed='{}']".format(" ".join(seed)))
    print(out)
    print()

print("Unique states   :", len(mc.trans))
print("Avg branching   : {:.2f}".format(np.mean([len(v) for v in mc.trans.values()])))
print("Max branching   :", max(len(v) for v in mc.trans.values()))


=== Markov Chain Order-3 Samples ===

[seed='machine learning algorithms']
machine learning algorithms can identify patterns in large datasets automatically. natural language processing enables computers to understand human speech and text. deep learning models require significant computational resources for training purposes. data science combines statistics mathematics and computer science for insights. artificial intelligence

[seed='artificial intelligence is']
artificial intelligence is transforming industries including healthcare finance and education. neural networks are inspired by the biological structure of the human brain. supervised learning uses labeled training data to build predictive classification models. unsupervised learning discovers hidden structure in unlabeled data distributions. reinforcement

[seed='the global economy']
the global economy faces challenges from inflation supply chain disruptions and conflict. space exploration has expanded human knowledge about 